# Train a residual CNN on one-step surge pairs
The five-convolution residual architecture and 1,000-epoch training settings are retained.
Use `24_hour_source.jld2`, which contains all 40 events required by the declared 1-28 / 29-34 / 35-40 split.
The former 72-hour control file has only 10 events. Forecasts are free rollouts from each event's initial state.
New model and plot files are written under `results/notebooks/base_pretraining/`.

In [ ]:
using Pkg
repository = @__DIR__
while !isfile(joinpath(repository, "src", "Surge.jl"))
    parent = dirname(repository)
    parent == repository && error("Open this notebook from within the repository.")
    global repository = parent
end
Pkg.activate(joinpath(repository, "Starting_networks"))
include(joinpath(repository, "src", "Surge.jl"))
import .Surge
using Flux, Random, Statistics, Plots, FFTW
include(joinpath(repository, "src", "plotting_utils.jl"))

In [ ]:
n_epochs = 1000
batch_size = 50
learning_rate = 1e-4
learning_rate_decay = 0.01
learning_rate_steps = 250
regularization_noise_level = 0.001f0
source_train_indices = collect(1:28)
source_val_indices = collect(29:34)
source_test_indices = collect(35:40)
# The original notebook had no fixed seed. Record a sampled seed for this run.
training_seed = rand(UInt32)
dataset_file = joinpath(repository, "Data", "24_hour_source.jld2")
output_directory = joinpath(repository, "results", "notebooks", "base_pretraining")
n_hidden = 16
n_filter = 3
n_input_channels = 5
n_output_channels = 2
n_state_channels = 2
training_settings = (
    epochs=n_epochs, batch_size=batch_size, learning_rate=learning_rate,
    learning_rate_decay=learning_rate_decay, learning_rate_steps=learning_rate_steps,
    learning_rate_decay_stages=3, noise_std=regularization_noise_level,
    checkpoint_interval=n_epochs, seed=training_seed,
)

In [ ]:
dataset = Surge.load_burgers_ensemble(basename(dataset_file);
    data_path=dirname(dataset_file), relative_times=true)

In [ ]:
training_pairs = Surge.one_step_pairs(dataset, source_train_indices)
validation_pairs = Surge.one_step_pairs(dataset, source_val_indices)
test_pairs = Surge.one_step_pairs(dataset, source_test_indices)
@show size(training_pairs.input) size(validation_pairs.input) size(test_pairs.input)

In [ ]:
Random.seed!(training_seed)
model = Surge.build_base_cnn(n_input_channels, n_output_channels,
    n_hidden, n_filter, n_state_channels)
@show model

In [ ]:
state = Surge.train_model!(model, training_pairs, validation_pairs; settings=training_settings)
model = state.model
Surge.restore_trainable_parameters!(model, state.best_parameters)
@show state.best_epoch state.best_loss
@show Surge.one_step_loss(model, test_pairs, batch_size)

In [ ]:
mkpath(output_directory)
model_save_path = joinpath(output_directory, "base_pretrained.bson")
Surge.save_model(model_save_path, model)

In [ ]:
run = (directory=output_directory, result=(history=state.history,))
mkpath(joinpath(output_directory, "evaluation"))
display(ExperimentPlots.plot_losses(run))

In [ ]:
event_index = first(source_test_indices)
event = dataset.solution[event_index]
forecast = Surge.forecast_event(Surge.effective_model(model), event, dataset.params)

In [ ]:
snapshots = plot(; xlabel="Cell index", ylabel="Water level [m]")
for step in (1, 11, 51)
    plot!(snapshots, forecast.prediction[:, 1, step]; label="Step $step")
end
display(snapshots)

In [ ]:
frame_indices = eachindex(dataset.times)
animation = @animate for step in frame_indices
    panels = [plot(grid, hcat(forecast.target[eachindex(grid), channel, step],
        forecast.prediction[eachindex(grid), channel, step]); label=["Reference" "Prediction"],
        xlabel="Position [m]", ylabel=label, title="Forecast $(round(dataset.times[step] / 3600; digits=2)) h")
        for (channel, grid, label) in ((1, dataset.grid.h, "h [m]"), (2, dataset.grid.u, "u [m/s]"))]
    plot(panels...; layout=(2, 1), size=(850, 600))
end
gif(animation, joinpath(output_directory, "forecast.gif"); fps=12)

In [ ]:
# Preserve the original spatial mean squared-error calculation, including h padding.
errors = dropdims(mean(abs2, forecast.prediction .- forecast.target; dims=1); dims=1)
for channel in 1:n_state_channels
    display(plot(dataset.times ./ 3600, errors[channel, :]; xlabel="Forecast [hours]",
        ylabel="MSE", title="Prediction error, channel $channel", label="Prediction"))
end

In [ ]:
# Spatial power spectra for the final held-out event; no additional training is performed.
last_event = dataset.solution[last(source_test_indices)]
last_forecast = Surge.forecast_event(Surge.effective_model(model), last_event, dataset.params)
truth, prediction = last_forecast.target, last_forecast.prediction
println("Test rollout MSE: ", Flux.mse(prediction, truth))
grid_spacing = dataset.params.dx

function mean_spatial_psd(values, channel)
    n_grid, _, n_steps = size(values)
    accumulated = zeros(Float64, div(n_grid, 2) + 1)
    for step in 1:n_steps
        signal = Float64.(values[:, channel, step]) .- Float64(mean(values[:, channel, step]))
        accumulated .+= abs.(rfft(signal)).^2 ./ n_grid
    end
    return accumulated ./ n_steps
end

for channel in 1:n_state_channels
    reference_psd = mean_spatial_psd(truth, channel)
    predicted_psd = mean_spatial_psd(prediction, channel)
    frequencies = (0:length(reference_psd)-1) ./ (size(truth, 1) * grid_spacing)
    positive = 2:length(frequencies)
    display(plot(frequencies[positive], hcat(reference_psd[positive], predicted_psd[positive]);
        xscale=:log10, yscale=:log10, label=["Truth" "Prediction"], xlabel="Wavenumber",
        ylabel="Power", title="Mean spatial PSD, channel $channel"))
    display(plot(frequencies[positive], (predicted_psd ./ reference_psd)[positive];
        xscale=:log10, yscale=:log10, label="Prediction/Truth", xlabel="Wavenumber", ylabel="Power ratio"))
    display(plot(frequencies[positive], abs.(predicted_psd .- reference_psd)[positive];
        xscale=:log10, yscale=:log10, label="Absolute PSD error", xlabel="Wavenumber", ylabel="Power error"))
end